# CH08 — Árboles binarios de búsqueda (BST)

Material del curso basado en Goodrich, Tamassia & Goldwasser — Sección 11.1.

Este cuaderno continúa `ch08_teoria.ipynb`: se asume que ya conoces el ADT de árbol binario y el recorrido **inorden** (secciones 8.2, 8.3 y 8.6).

**Contenido**

- 8.8 Árbol binario de búsqueda — definición y ADT
- 8.9 Búsqueda
- 8.10 Inserción
- 8.11 Eliminación
- 8.12 Altura y rendimiento
- 8.13 Para la clase: implementa el BST
- 8.14 Ejercicios propuestos


## 8.8 Árbol binario de búsqueda — definición

Un **árbol binario de búsqueda** (*Binary Search Tree*, BST) es un árbol binario que guarda elementos **comparables** (números, cadenas, ...) y cumple la **propiedad de orden** en **cada** nodo `v`:

- Todos los elementos del **subárbol izquierdo** de `v` son **menores** que el elemento de `v`.
- Todos los elementos del **subárbol derecho** de `v` son **mayores** que el elemento de `v`.

En este cuaderno los elementos son **distintos**: insertar un elemento que ya está no hace nada.

**Ojo:** la propiedad habla de **subárboles completos**, no solo de los hijos. En la figura, el árbol de la derecha **no** es un BST: cada nodo está bien ubicado respecto a su padre (`55 > 30`, así que va a la derecha de `30`), pero `55` quedó dentro del subárbol **izquierdo** de `50`, y `55 > 50`.

**Consecuencia clave:** el recorrido **inorden** (Izq → Raíz → Der) de un BST visita los elementos en **orden ascendente**. En el árbol válido de la figura: `20 30 35 40 50 60 65 70 80`. Esa es la razón de ser del BST: mantener datos **ordenados** y aun así poder insertar y eliminar sin reacomodar todo — en un arreglo ordenado, insertar obliga a correr los elementos una posición.


![BST válido y árbol que no es BST](../assets/ch08_bst_definicion.png)

### ADT — Árbol binario de búsqueda

Además del ADT de árbol binario (`root`, `parent`, `left`, `right`, ...), un BST agrega operaciones que **aprovechan el orden**. En la tabla, $h$ es la **altura** del árbol y $n$ el número de nodos.

| Método | Descripción | $O$ |
|---|---|---|
| `search(k)` | Nodo que contiene `k` (`None` si no está) | $O(h)$ |
| `insert(k)` | Inserta `k` como hoja en su lugar (si ya está, no hace nada) | $O(h)$ |
| `delete(k)` | Elimina `k` manteniendo la propiedad de orden (`KeyError` si no está) | $O(h)$ |
| `find_min()` | Nodo con el menor elemento | $O(h)$ |
| `find_max()` | Nodo con el mayor elemento | $O(h)$ |
| `inorder()` | Genera los nodos en orden ascendente | $O(n)$ |

Todas las operaciones de la tabla, salvo `inorder`, recorren **un solo camino** desde la raíz hacia abajo: su costo depende de la altura $h$, no del número de nodos $n$ (sección 8.12).


## 8.9 Búsqueda

Para buscar `k` se arranca en la raíz y en cada nodo `v` se toma **una sola decisión**:

- Si `k == v`: se encontró.
- Si `k < v`: `k` solo puede estar en el subárbol **izquierdo** → se baja a la izquierda.
- Si `k > v`: `k` solo puede estar en el subárbol **derecho** → se baja a la derecha.
- Si hay que bajar a un hijo que **no existe** (`None`): `k` no está en el árbol.

Cada comparación **descarta un subárbol completo**, igual que la búsqueda binaria descarta media lista en cada paso. La búsqueda sigue un único camino desde la raíz hasta, a lo más, una hoja: $O(h)$.

En la figura de la sección 8.10:

- `search(65)`: `65 > 50` → derecha; `65 < 70` → izquierda; `65 > 60` → derecha; encontrado.
- `search(25)`: `25 < 50` → izquierda; `25 < 30` → izquierda; `25 > 20` → derecha, pero `20` no tiene hijo derecho → **no está**.

**Mínimo y máximo** son el caso extremo de la búsqueda: el menor elemento se encuentra bajando **siempre a la izquierda** hasta que no haya hijo izquierdo; el mayor, **siempre a la derecha**. En el árbol de la sección 8.8: mínimo `20`, máximo `80`.


## 8.10 Inserción

Insertar `k` es **buscar `k` y colgarlo donde la búsqueda falla**: el hijo `None` en el que termina la búsqueda es el único lugar donde `k` respeta la propiedad de orden. Por eso **todo elemento nuevo entra como hoja** y los nodos que ya estaban nunca se mueven.

En la figura, `insert(45)`: `45 < 50` → izquierda; `45 > 30` → derecha; `45 > 40` → derecha, pero `40` no tiene hijo derecho → `45` queda como **hijo derecho de `40`**.

**El orden de inserción determina la forma del árbol.** Los mismos elementos insertados en otro orden producen otro árbol (con el mismo inorden). El primer elemento insertado siempre queda como raíz.


![Búsqueda e inserción en un BST](../assets/ch08_bst_buscar_insertar.png)

## 8.11 Eliminación

Eliminar es la operación delicada: hay que quitar el nodo **sin romper** la propiedad de orden. Primero se busca el nodo `v` que contiene `k`; lo que sigue depende de cuántos hijos tiene `v`:

| Caso | Situación | Qué se hace |
|---|---|---|
| **1** | `v` es **hoja** | Se quita: su padre pasa a apuntar a `None`. |
| **2** | `v` tiene **un hijo** | El hijo **sube** a ocupar el lugar de `v` (queda enlazado con el padre de `v`). |
| **3** | `v` tiene **dos hijos** | Se copia en `v` el elemento de su **predecesor** — el **mayor** del subárbol izquierdo — y se elimina el nodo del predecesor. |

**Por qué el predecesor funciona en el caso 3:** es mayor que todo lo que queda en el subárbol izquierdo y menor que todo lo del derecho, así que puesto en el lugar de `v` la propiedad de orden se mantiene. Además, el predecesor **nunca tiene hijo derecho** (si lo tuviera, ese hijo sería mayor y el predecesor no sería el máximo), así que eliminar su nodo cae en el caso 1 o en el 2: **el caso 3 se reduce a los otros dos**.

En la figura, al eliminar `50` (la raíz, con dos hijos) su predecesor es `40`: el `40` pasa a la raíz y su hijo `35` sube al lugar que dejó.

También funciona usar el **sucesor** (el menor del subárbol derecho). Goodrich usa el predecesor y aquí seguimos esa convención.


![Los tres casos de eliminación](../assets/ch08_bst_eliminacion.png)

## 8.12 Altura y rendimiento

`search`, `insert`, `delete`, `find_min` y `find_max` cuestan $O(h)$. Cuánto vale $h$ depende de la **forma** del árbol, y la forma depende del **orden de inserción**:

| Caso | Forma | Altura $h$ | Costo |
|---|---|---|---|
| Mejor caso | Balanceado: cada nivel (casi) lleno | $\approx \log_2 n$ | $O(\log n)$ |
| Peor caso | Degenerado: una cadena, cada nodo con un solo hijo | $n - 1$ | $O(n)$ |

El peor caso no es raro: basta insertar los elementos **ya ordenados** (`1, 2, ..., 10`). Cada elemento nuevo es mayor que todos los anteriores, así que siempre cae a la derecha del último y el árbol se vuelve, en la práctica, una lista enlazada. Con $n = 1\,000\,000$ elementos, un árbol balanceado encuentra cualquiera en unas 20 comparaciones; el degenerado puede necesitar un millón.

La figura muestra los mismos 10 elementos insertados en orden en un BST común (izquierda) y en un árbol **AVL** (derecha), que **se rebalancea solo** después de cada inserción para garantizar $h = O(\log n)$. Los árboles balanceados (AVL, rojo-negro, splay) se estudian en el capítulo 11.


![BST degenerado vs AVL balanceado](../assets/ch11_bst_vs_avl.png)

## 8.13 Para la clase: implementa el BST

Implementa `BinarySearchTree` con **nodos enlazados directos**, igual que `LinkedBinaryTree` en `ch08_teoria` (sección 8.5): `_Node` anidado, sin `Position`, y los métodos reciben y retornan **nodos**. Los elementos son las propias claves (números). Los nombres de los métodos se mantienen en inglés, igual que en el ADT de la sección 8.8.

Ya vienen implementados el ADT de árbol binario (`root`, `parent`, `left`, `right`, `num_children`, `is_leaf`, `len`, `is_empty`), `height()` e `inorder()` (lo resolviste en la sección 8.6).

Completa los métodos marcados con `pass`: `search`, `insert`, `find_min`, `find_max` y `delete`.

**Pistas**

- `search` e `insert` se pueden escribir con un `while` que baja por el árbol o de forma recursiva. Al crear un nodo, enlázalo en **ambas direcciones**: el padre apunta al hijo (`_left` o `_right`) y el hijo al padre (`_parent`).
- Para `delete` conviene un método auxiliar `_subtree_max(node)` que encuentre el predecesor. Resuelve primero los casos 1 y 2; el caso 3 copia en `v` el elemento del predecesor y luego elimina el nodo del predecesor con esos mismos casos.
- Si el nodo que se quita es la **raíz**, actualiza `self._root`. No olvides actualizar `self._size` en `insert` y en `delete`.


In [ ]:
class BinarySearchTree:
    """Árbol binario de búsqueda con nodos enlazados directos (sin Position)."""

    class _Node:
        """Nodo interno: elemento, padre, hijo izquierdo, hijo derecho."""
        __slots__ = ('_element', '_parent', '_left', '_right')

        def __init__(self, element, parent=None, left=None, right=None):
            self._element = element
            self._parent = parent
            self._left = left
            self._right = right

        def __repr__(self):
            return f'Node({self._element})'

    def __init__(self):
        self._root = None
        self._size = 0

    # --- ADT de árbol binario (ya implementado) ---
    def __len__(self):
        """Return the total number of nodes in the tree."""
        return self._size

    def is_empty(self):
        """Return True if the tree does not contain any nodes."""
        return self._size == 0

    def root(self):
        """Return the root node of the tree (or None if tree is empty)."""
        return self._root

    def parent(self, node):
        """Return the node's parent (or None if node is the root)."""
        return node._parent

    def left(self, node):
        """Return the node's left child (or None if no left child)."""
        return node._left

    def right(self, node):
        """Return the node's right child (or None if no right child)."""
        return node._right

    def num_children(self, node):
        """Return the number of children of node (0, 1, or 2)."""
        return (node._left is not None) + (node._right is not None)

    def is_leaf(self, node):
        """Return True if node does not have any children."""
        return node._left is None and node._right is None

    def height(self, node=None):
        """Return the height of the subtree rooted at node (whole tree by default; -1 if empty)."""
        if node is None:
            node = self._root
        if node is None:
            return -1
        if self.is_leaf(node):
            return 0
        return 1 + max(self.height(c) for c in (node._left, node._right) if c is not None)

    def inorder(self):
        """Generate an inorder iteration of the tree's nodes (ascending order in a BST)."""
        if self._root is not None:
            yield from self._subtree_inorder(self._root)

    def _subtree_inorder(self, node):
        """Generate an inorder iteration of the subtree rooted at node."""
        if node._left is not None:
            yield from self._subtree_inorder(node._left)
        yield node
        if node._right is not None:
            yield from self._subtree_inorder(node._right)

    # --- Operaciones de BST ---
    def search(self, k):
        """Return the node that contains k (or None if k is not in the tree)."""
        pass

    def insert(self, k):
        """Insert k as a new leaf in its place and return its node.

        If k is already in the tree, do nothing and return the existing node.
        """
        pass

    def find_min(self):
        """Return the node with the smallest element (or None if tree is empty)."""
        pass

    def find_max(self):
        """Return the node with the largest element (or None if tree is empty)."""
        pass

    def delete(self, k):
        """Remove k from the tree keeping the BST property. Raise KeyError if k is not present."""
        pass


In [ ]:
# --- Tests: construcción e inorden (árbol de la sección 8.8) ---
T = BinarySearchTree()
for k in [50, 30, 70, 20, 40, 60, 80, 35, 65]:
    T.insert(k)

print(len(T))                                                   # 9
print(T.root()._element)                                        # 50
print(T.left(T.root())._element, T.right(T.root())._element)    # 30 70
print([n._element for n in T.inorder()])                        # [20, 30, 35, 40, 50, 60, 65, 70, 80]
print(T.height())                                               # 3


In [ ]:
# --- Tests: búsqueda, mínimo y máximo ---
print(T.search(65)._element)                                    # 65
print(T.search(65)._parent._element)                            # 60
print(T.search(25))                                             # None  (no está)
print(T.search(50) is T.root())                                 # True
print(T.find_min()._element, T.find_max()._element)             # 20 80
print(BinarySearchTree().search(10))                            # None  (árbol vacío)
print(BinarySearchTree().find_min())                            # None  (árbol vacío)


In [ ]:
# --- Tests: inserción ---
nodo = T.insert(40)                                             # 40 ya está: no se duplica
print(nodo is T.search(40), len(T))                             # True 9

nuevo = T.insert(45)                                            # figura de la sección 8.10
print(nuevo._parent._element, len(T))                           # 40 10
print(T.right(T.search(40)) is nuevo, nuevo._parent is T.search(40))   # True True
print(T.is_leaf(nuevo))                                         # True  (todo elemento nuevo entra como hoja)


In [ ]:
# --- Tests: eliminación (los tres casos de la sección 8.11) ---
T = BinarySearchTree()
for k in [50, 30, 70, 20, 40, 60, 80, 35, 65]:
    T.insert(k)

T.delete(20)                                                    # caso 1: hoja
print(T.left(T.search(30)), len(T))                             # None 8

T.delete(60)                                                    # caso 2: un hijo (65 sube)
print(T.left(T.search(70))._element)                            # 65
print(T.search(65)._parent._element)                            # 70

T.delete(50)                                                    # caso 3: dos hijos (lo reemplaza el predecesor 40)
print(T.root()._element)                                        # 40
print(T.right(T.search(30))._element)                           # 35
print([n._element for n in T.inorder()], len(T))                # [30, 35, 40, 65, 70, 80] 6

try:
    T.delete(99)
except KeyError:
    print('KeyError')                                           # KeyError

U = BinarySearchTree()
U.insert(10)
U.delete(10)                                                    # se elimina la raíz y el árbol queda vacío
print(len(U), U.root())                                         # 0 None


In [ ]:
# --- Tests: el orden de inserción determina la altura (sección 8.12) ---
D = BinarySearchTree()
for k in range(1, 11):                                          # 1, 2, ..., 10 ya ordenados
    D.insert(k)

B = BinarySearchTree()
for k in [5, 2, 8, 1, 3, 7, 9, 4, 6, 10]:                       # los mismos 10 elementos, otro orden
    B.insert(k)

print(D.height(), B.height())                                   # 9 3
print([n._element for n in D.inorder()] == [n._element for n in B.inorder()])   # True  (mismo inorden)


## 8.14 Ejercicios propuestos

1. **Trazar a mano.** Inserta en un BST vacío `8, 3, 10, 1, 6, 14, 4, 7, 13`. Dibuja el árbol y escribe su inorden y su altura. Luego elimina `3` y dibuja el resultado.
2. **¿Es un BST?** Escribe `es_bst(T)` para un `LinkedBinaryTree` cualquiera. Cuidado con la trampa de la sección 8.8: no basta comparar cada nodo con sus hijos. *Pista:* pásale a la recursión el rango `(minimo, maximo)` permitido para cada subárbol.
3. **Piso.** Agrega `find_le(k)`: el mayor elemento **menor o igual** que `k` (o `None` si no hay), en $O(h)$. En el árbol de la sección 8.8, `find_le(62)` es `60`.
4. **Consulta por rango.** Agrega un generador `find_range(a, b)` que produzca, en orden, los elementos `x` con `a <= x < b`, sin entrar a subárboles que no pueden tener resultados.
5. **k-ésimo menor.** Escribe `kesimo(T, k)` que retorne el k-ésimo menor elemento, deteniendo el recorrido inorden apenas lo encuentre.
6. **¿Qué orden de inserción?** ¿En qué orden hay que insertar `1, 2, ..., 7` para obtener un árbol de altura 2? ¿Hay un solo orden que lo logre?
